In [34]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    TaskType
)

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [36]:
model_id = "microsoft/phi-2"
output_dir = "./phi2-qlora-finetuned-med/"

In [37]:
# Configure QLoRA parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [38]:
# Load the base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [39]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [40]:
# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                       # Rank
    lora_alpha=16,             # Alpha parameter
    lora_dropout=0.1,          # Dropout probability
    target_modules=["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"],  # Verify these match Phi-2's architecture
    bias="none",
    inference_mode=False,
)

In [41]:
# Prepare model for QLoRA fine-tuning
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)


In [42]:
dataset = Dataset.from_csv("final_dataset.csv")
dataset_dict = dataset.train_test_split(
    test_size=0.1,
    seed=42,
    shuffle=True
)

In [43]:
dataset_dict = DatasetDict({
    'train': dataset_dict['train'],
    'validation': dataset_dict['test']
})

In [44]:
def create_conversation_pairs(hf_dataset):
    """
    Given a HuggingFace Dataset with columns 'Question' and 'Answer',
    returns a list of dicts with keys 'instruction' and 'output'.
    """
    pairs = []
    for example in hf_dataset:
        pairs.append({
            "instruction": example["input"],
            "output": example["output"]
        })
    return pairs

In [45]:
conversation_pairs = create_conversation_pairs(dataset_dict['train'])
print(f"Number of pairs: {len(conversation_pairs)}")
print("First pair:", conversation_pairs[0])

Number of pairs: 76047
First pair: {'instruction': 'What is suggested by the presence of a new murmur after a prosthetic valve has been implanted?', 'output': 'The presence of a new murmur after a prosthetic valve has been implanted is suggestive of prosthetic valve dysfunction. A new murmur can indicate that there is a problem with the valve, such as a leak or a blockage, which can lead to reduced blood flow and other complications. In some cases, additional testing, such as an echocardiogram or cardiac catheterization, may be needed to confirm the diagnosis and determine the best course of treatment.'}


In [46]:
conversation_pairs_validation = create_conversation_pairs(dataset_dict['validation'])
print(f"Number of pairs: {len(conversation_pairs_validation)}")
print("First pair:", conversation_pairs_validation[0])

Number of pairs: 8450
First pair: {'instruction': 'What is subacute sclerosing panencephalitis (SSPE), and what causes it?', 'output': 'Subacute sclerosing panencephalitis (SSPE) is a rare, progressive, and usually fatal neurological disorder that occurs as a result of persistent infection of the brain by the measles virus. The virus causes inflammation and damage to the brain, leading to symptoms such as seizures, dementia, and loss of motor function. SSPE typically develops several years after a person has had measles, and is more common in individuals who contracted the virus at a young age. There is currently no cure for SSPE, and treatment is mainly focused on managing the symptoms.'}


In [47]:
from datasets import Dataset
train_dataset = Dataset.from_list(conversation_pairs)

In [48]:
# Function to tokenize the dataset
def tokenize_function(examples):
    # Format: <|user|>instruction<|assistant|>output<|endoftext|>
    prompts = []
    labels = []
    
    for instruction, output in zip(examples["instruction"], examples["output"]):
        # Create the prompt
        prompt = f"<|user|>{instruction}<|assistant|>{output}<|endoftext|>"
        prompts.append(prompt)
    
    tokenized_inputs = tokenizer(
        prompts,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt",
    )
    
    # Create labels (same as input_ids for causal LM)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].clone()
    
    return tokenized_inputs

In [49]:
# Tokenize the datasets
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["instruction", "output"],
)

Map:   0%|          | 0/76047 [00:00<?, ? examples/s]

In [50]:
val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["instruction", "output"],
)

Map:   0%|          | 0/8450 [00:00<?, ? examples/s]

In [51]:
# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [52]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir="./logs",
    logging_steps=100,
    fp16=True,
    report_to="tensorboard",
    gradient_checkpointing=True,
)

In [53]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)


In [54]:
trainer.train()

  0%|          | 0/23765 [00:00<?, ?it/s]

{'loss': 1.7078, 'grad_norm': 0.3364907503128052, 'learning_rate': 4e-05, 'epoch': 0.02}
{'loss': 1.3527, 'grad_norm': 0.41641950607299805, 'learning_rate': 8e-05, 'epoch': 0.04}
{'loss': 1.2517, 'grad_norm': 0.48557785153388977, 'learning_rate': 0.00012, 'epoch': 0.06}
{'loss': 1.1951, 'grad_norm': 0.4875071048736572, 'learning_rate': 0.00016, 'epoch': 0.08}
{'loss': 1.1808, 'grad_norm': 0.38681256771087646, 'learning_rate': 0.0002, 'epoch': 0.11}


  0%|          | 0/4225 [00:00<?, ?it/s]

{'eval_loss': 1.645485520362854, 'eval_runtime': 1485.8774, 'eval_samples_per_second': 5.687, 'eval_steps_per_second': 2.843, 'epoch': 0.11}


c:\Users\vasal\anaconda3\envs\mnist_env\lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


...

  0%|          | 0/4225 [00:00<?, ?it/s]

{'eval_loss': 1.4663922786712646, 'eval_runtime': 1472.9203, 'eval_samples_per_second': 5.737, 'eval_steps_per_second': 2.868, 'epoch': 4.94}


c:\Users\vasal\anaconda3\envs\mnist_env\lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 0.9842, 'grad_norm': 0.33112937211990356, 'learning_rate': 1.4872125510423383e-06, 'epoch': 4.97}
{'loss': 0.9713, 'grad_norm': 0.35899433493614197, 'learning_rate': 6.27552116913819e-07, 'epoch': 4.99}
{'train_runtime': 251980.4506, 'train_samples_per_second': 1.509, 'train_steps_per_second': 0.094, 'train_loss': 1.0186508692366787, 'epoch': 5.0}


TrainOutput(global_step=23765, training_loss=1.0186508692366787, metrics={'train_runtime': 251980.4506, 'train_samples_per_second': 1.509, 'train_steps_per_second': 0.094, 'total_flos': 1.5476633158680576e+18, 'train_loss': 1.0186508692366787, 'epoch': 5.0})

In [55]:
# Save the fine-tuned model
peft_model_path = os.path.join(output_dir, "peft_model")
model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(output_dir)

from peft import PeftConfig
config = PeftConfig.from_pretrained(peft_model_path)
config.save_pretrained(peft_model_path)

print(f"Model saved to {peft_model_path}")
print(f"Tokenizer saved to {output_dir}")

Model saved to ./phi2-qlora-finetuned-med/peft_model
Tokenizer saved to ./phi2-qlora-finetuned-med/


In [60]:
# Function for inference
def generate_response(instruction, model, tokenizer, max_length=256):
    prompt = f"<|user|>{instruction}<|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract only the assistant's response
    response = response.split("<|assistant|>")[1].split("<|endoftext|>")[0].strip()
    return response

In [57]:
# Test the model with a sample question
test_instruction = " how long will 6 mgs of suboxone block opiates?"
response = generate_response(test_instruction, model, tokenizer)
print(f"Instruction: {test_instruction}")
print(f"Response: {response}")

c:\Users\vasal\anaconda3\envs\mnist_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\vasal\anaconda3\envs\mnist_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\vasal\anaconda3\envs\mnist_env\lib\site-packages\torch\utils\checkpoint.py:92: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Instruction:  how long will 6 mgs of suboxone block opiates?
Response: Suboxone is a partial agonist, which means that it binds to the opioid receptors in the brain and produces a similar effect to opioids, but with a lower potency. This means that it can help to reduce the symptoms of opioid withdrawal, but it is not as strong as other opioids. The duration of action of suboxone varies depending on the individual and the specific formulation of the medication. In general, suboxone is effective for up to 12 hours, but the effects may last longer in some individuals. It is important to follow the prescribed dosage and schedule to ensure that the medication is effective and to avoid the risk of overdose or other adverse effects. If you have any questions or concerns about suboxone, it is important to talk to your healthcare provider or a healthcare professional who specializes in addiction treatment. They can provide you with more information and guidance on how to use suboxone safely an

In [58]:
# Test the model with a sample question
test_instruction = " How can I treat my burnt tongue?"
response = generate_response(test_instruction, model, tokenizer)
print(f"Instruction: {test_instruction}")
print(f"Response: {response}")

Instruction:  How can I treat my burnt tongue?
Response: Treatment for a burnt tongue depends on the severity of the burn. For mild burns, you can rinse your mouth with cool water or milk to soothe the pain. You can also suck on ice chips or use a numbing gel or spray. Avoid eating or drinking anything hot or spicy, as this can make the pain worse. For more severe burns, you may need to see a doctor. They may prescribe pain medication or recommend a topical anesthetic to help with the pain. In some cases, a burn on the tongue may require a tetanus shot or other medical treatment. It is important to seek medical attention if you have a severe burn on your tongue, as it can lead to infection or other complications. In general, it is best to avoid burning your tongue in the first place, by being careful when eating or drinking hot or spicy foods. If you do burn your tongue, take steps to soothe the pain and prevent infection. If you have any concerns about the severity of your burn, it is

In [59]:
# Test the model with a sample question
test_instruction = " I am pregnent, I am getting UTI consistenly, what to do?"
response = generate_response(test_instruction, model, tokenizer)
print(f"Instruction: {test_instruction}")
print(f"Response: {response}")

Instruction:  I am pregnent, I am getting UTI consistenly, what to do?
Response: Urinary tract infections (UTIs) are common in pregnancy. UTIs are caused by bacteria that enter the urinary tract. Most UTIs are caused by E. coli, a type of bacteria that is normally found in the intestines. UTIs are more common in pregnant women because the growing uterus puts pressure on the bladder and urethra, making it easier for bacteria to enter the urinary tract. UTIs can cause pain or burning during urination, frequent urination, and cloudy or bloody urine. Treatment for UTIs in pregnancy usually involves antibiotics. It is important to treat UTIs promptly to prevent complications such as preterm labor and low birth weight. Your healthcare provider can help you determine the best treatment for your UTI.  Learn more about UTIs in pregnancy.  Learn more about UTI symptoms.  Learn more about UTI treatment.  Learn more about UTI prevention.  Learn more about UTI in children.  Learn more about UTI in 

In [61]:
# Test the model with a sample question
test_instruction = " I am pregnent, I am getting UTI consistenly, what to do?"
response = generate_response(test_instruction, model, tokenizer)
print(f"Instruction: {test_instruction}")
print(f"Response: {response}")

Instruction:  I am pregnent, I am getting UTI consistenly, what to do?
Response: If you have a urinary tract infection (UTI), your doctor will prescribe an antibiotic to kill the bacteria. You may need to take the antibiotic for several days. If you do not take the full course of antibiotics, the infection may return.  If you have a UTI, you should drink plenty of fluids to help flush out the bacteria. You should also urinate often.  If you have a UTI, you should drink plenty of fluids to help flush out the bacteria. You should also urinate often.  If you have a UTI, you should drink plenty of fluids to help flush out the bacteria. You should also urinate often.  If you have a UTI, you should drink plenty of fluids to help flush out the bacteria. You should also urinate often.  If you have a UTI, you should drink plenty of fluids to help flush out the bacteria. You should also urinate often.  If you have a UTI, you should drink plenty of fluids to help flush out the bacteria. You shoul